### NanoGPT Implementation
- Implementation of Karpathy's lecture on building GPT from scratch: https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=10

- Google collab from lecture: https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=O6medjfRsLD9


### Get input data

In [1]:
# import urllib.request

# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# urllib.request.urlretrieve(url, "input.txt")

### Build data processing and model in pieces

In [2]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

print("".join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}

# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

print(encode("my name is zeal"))
print(decode(encode("my name is zeal")))

[51, 63, 1, 52, 39, 51, 43, 1, 47, 57, 1, 64, 43, 39, 50]
my name is zeal


In [7]:
# train and test splits
import torch

data = torch.tensor(encode(text), dtype=torch.long)

# 90% training and 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Data size: Train = {len(train_data)} | Val = {len(val_data)}")

Data size: Train = 1003854 | Val = 111540


In [8]:
# extract one block of training examples
# a single block contains block_size number of examples

block_size = 8  # same as context_length

x = train_data[:block_size]  # single block
y = train_data[1 : block_size + 1]


print("Single block of example packs block_size number of examples")
for i in range(block_size):
    context = x[: i + 1]
    target = y[i]
    print(f"Example {i} --> context = {context} and target = {target} ")


Single block of example packs block_size number of examples
Example 0 --> context = tensor([18]) and target = 47 
Example 1 --> context = tensor([18, 47]) and target = 56 
Example 2 --> context = tensor([18, 47, 56]) and target = 57 
Example 3 --> context = tensor([18, 47, 56, 57]) and target = 58 
Example 4 --> context = tensor([18, 47, 56, 57, 58]) and target = 1 
Example 5 --> context = tensor([18, 47, 56, 57, 58,  1]) and target = 15 
Example 6 --> context = tensor([18, 47, 56, 57, 58,  1, 15]) and target = 47 
Example 7 --> context = tensor([18, 47, 56, 57, 58,  1, 15, 47]) and target = 58 


In [ ]:
# data loader
block_size = 8
batch_size = 4


# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    return x, y


xb, yb = get_batch(split="train")

print(f"Inputs: {x.shape} \n {x}")
print(f"Targets: {y.shape} \n {y}")

# -------------------------------------------------------------------------------------------------
# visualize every example packed in these four batches for our DECODER transformer block
eg = 0
for batch in range(batch_size):
    for time in range(block_size):
        context = xb[batch, 0 : time + 1]
        target = yb[batch, time]
        print(f"Example {eg} --> context = {context} and target = {target}")
        eg += 1


Inputs: torch.Size([8]) 
 tensor([18, 47, 56, 57, 58,  1, 15, 47])
Targets: torch.Size([8]) 
 tensor([47, 56, 57, 58,  1, 15, 47, 58])
Example 0 --> context = tensor([40]) and target = 47
Example 1 --> context = tensor([40, 47]) and target = 58
Example 2 --> context = tensor([40, 47, 58]) and target = 1
Example 3 --> context = tensor([40, 47, 58,  1]) and target = 39
Example 4 --> context = tensor([40, 47, 58,  1, 39]) and target = 52
Example 5 --> context = tensor([40, 47, 58,  1, 39, 52]) and target = 42
Example 6 --> context = tensor([40, 47, 58,  1, 39, 52, 42]) and target = 1
Example 7 --> context = tensor([40, 47, 58,  1, 39, 52, 42,  1]) and target = 47
Example 8 --> context = tensor([1]) and target = 50
Example 9 --> context = tensor([ 1, 50]) and target = 59
Example 10 --> context = tensor([ 1, 50, 59]) and target = 57
Example 11 --> context = tensor([ 1, 50, 59, 57]) and target = 58
Example 12 --> context = tensor([ 1, 50, 59, 57, 58]) and target = 11
Example 13 --> context =

In [13]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

In [50]:
# start with bigram model. Bigram model uses a (vocab_size, vocab_size) embedding matrix
# passing "x" of size (3,4) to this matrix --> bigram_embedding_matrix(x) --> it will return a (3,4,vocab_size) output
# where, each element of x is now embedded into vocab_size dimensions


class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, targets=None):
        # x is of size (B,T), where each element is a character index from the vocabulary
        # target is of size (B,T)

        logits = self.token_embedding_table(x)  # (B,T,vocab_size) after passing through the embedding table

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, x, max_new_tokens):
        # generate one token at a time
        for i in range(max_new_tokens):
            # forward pass through the model
            logits, _ = self.forward(x)  # logits is (B,T,vocab_size)
            # focus only on the last timestep; no dependence on past here
            logits = logits[:, -1, :]  # (B,vocab_size)
            # convert logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample next token from probability distribution
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append next_token to the original context
            x = torch.cat((x, next_token), dim=1)  # (B,T+1)
        return x


# call the model
model = BigramModel(vocab_size)
logits, loss = model(x=xb, targets=yb)
print(f"Loss = {loss}")

model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=10)
print(decode(model_output[0].tolist()))

Loss = 4.719639778137207

:RLhuj$ptt
